# Prayogshala end-to-end run - MLOps Assignment 1 (Heart Disease)
**Shankaraguru Ramdurg - 2025AE05523 - AIMLCZG523**

Runs the whole pipeline inside a BITS Prayogshala (Kubeflow) Jupyter notebook server, top to bottom:

1. environment check and clean install from `requirements.txt`
2. data download (falls back to the raw files shipped in the repo if UCI is blocked)
3. unit tests (pytest) and lint (flake8) - the same gates as the GitHub Actions pipeline
4. EDA, training with MLflow tracking and model registry
5. serving: starts the FastAPI service in the background, calls `/predict`, generates traffic, reads `/metrics`
6. optional: MLflow UI through the notebook proxy, Kubernetes permission check, Kubeflow Pipelines run

**How to use:** upload `heart-disease-mlops.zip` to the notebook server's home folder (or set `REPO_URL` below to your
GitHub repo), open this notebook and choose *Run All*. A CPU notebook is enough (about 3-5 minutes); no GPU is needed.

In [ ]:
# ---- settings -------------------------------------------------------------
REPO_URL = ""            # e.g. "https://github.com/<you>/heart-disease-mlops.git" (leave empty to use the uploaded zip)
RUN_KFP_PIPELINE = False # set True to also submit the Kubeflow pipeline (needs kfp + pipeline access)
API_PORT = 8000
MLFLOW_PORT = 5000

In [ ]:
import os, sys, subprocess, zipfile, platform, shutil, time, json
from pathlib import Path

def sh(cmd, cwd=None, check=True):
    """Run a shell command, stream a compact log, fail loudly like CI would."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    print(out[-4000:] if len(out) > 4000 else out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r

# locate / fetch the project
here = Path.cwd()
candidates = [here, here.parent, Path.home() / "heart-disease-mlops"]
ROOT = next((p for p in candidates if (p / "params.yaml").exists()), None)
if ROOT is None and REPO_URL:
    sh(f"git clone --depth 1 {REPO_URL} {Path.home() / 'heart-disease-mlops'}")
    ROOT = Path.home() / "heart-disease-mlops"
if ROOT is None:
    z = next(iter(sorted(Path.home().glob("heart-disease-mlops*.zip"))), None)
    assert z, "Upload heart-disease-mlops.zip to your home folder or set REPO_URL"
    zipfile.ZipFile(z).extractall(Path.home())
    ROOT = Path.home() / "heart-disease-mlops"
os.chdir(ROOT)
print("project:", ROOT)
print("python :", platform.python_version(), "| cpus:", os.cpu_count())
print("gpu    :", "yes" if shutil.which("nvidia-smi") else "no (not needed)")
print("kubectl:", shutil.which("kubectl") or "not installed")

## 1. Clean install from the pinned requirements

In [ ]:
sh(f"{sys.executable} -m pip install -q -r requirements.txt")
sh(f"{sys.executable} -m pip install -q -e .")
sys.path.insert(0, str(ROOT / "src"))
import sklearn, mlflow, fastapi
print("sklearn", sklearn.__version__, "| mlflow", mlflow.__version__, "| fastapi", fastapi.__version__)

## 2. Data

In [ ]:
sh(f"{sys.executable} scripts/download_data.py --skip-if-present")
sh("cd data/raw && sha256sum -c CHECKSUMS.sha256")

## 3. Quality gates: lint + unit tests (same as CI)

In [ ]:
sh(f"{sys.executable} -m flake8 src tests scripts")
r = sh(f"{sys.executable} -m pytest -q --cov=cardiorisk --cov-report=term", check=True)

## 4. EDA and training with MLflow tracking

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{ROOT / 'mlflow.db'}"
sh(f"{sys.executable} -m cardiorisk.eda")
t0 = time.time()
sh(f"{sys.executable} -m cardiorisk.train")
print(f"training finished in {time.time() - t0:.0f}s")

In [ ]:
import pandas as pd
m = json.loads((ROOT / "reports/metrics.json").read_text())
print("champion:", m["champion"], "| decision threshold:", m["decision_threshold"])
display(pd.DataFrame({k: {x: v[x] for x in ["cv_roc_auc_mean","cv_accuracy_mean","cv_recall_mean","test_roc_auc","test_accuracy"]}
                      for k, v in m["models"].items()}).T)
pd.Series(m["champion_test"]).round(4)

In [ ]:
import mlflow
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
runs = mlflow.search_runs(experiment_names=["heart-disease-risk"])
print(len(runs), "runs logged;", runs["tags.run_type"].value_counts().to_dict())
client = mlflow.tracking.MlflowClient()
mv = client.get_model_version_by_alias("heart-disease-classifier", "champion")
print("registry: heart-disease-classifier version", mv.version, "alias=champion, run", mv.run_id)

### MLflow UI (optional)
Starts the UI in the background. In Kubeflow, if the notebook image has `jupyter-server-proxy`, open
`https://<prayogshala-host>/notebook/<your-namespace>/<notebook-name>/proxy/5000/`. Otherwise run
`kubectl port-forward` from a machine with cluster access, or copy `mlflow.db` + `mlartifacts/` to your laptop
and run `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
mlflow_ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", os.environ["MLFLOW_TRACKING_URI"],
                              "--host", "0.0.0.0", "--port", str(MLFLOW_PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("MLflow UI pid", mlflow_ui.pid, "on port", MLFLOW_PORT)

## 5. Serve the model and call the API

In [ ]:
import urllib.request, urllib.error
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "cardiorisk.serving.app:app", "--host", "0.0.0.0",
                        "--port", str(API_PORT)], cwd=ROOT, stdout=open(ROOT / "api.log", "w"), stderr=subprocess.STDOUT)
BASE = f"http://127.0.0.1:{API_PORT}"
for _ in range(60):
    try:
        if urllib.request.urlopen(BASE + "/ready", timeout=1).status == 200: break
    except Exception: time.sleep(1)
print(urllib.request.urlopen(BASE + "/ready").read().decode())

def post(path, body):
    req = urllib.request.Request(BASE + path, data=json.dumps(body).encode(), headers={"content-type": "application/json"})
    try:
        with urllib.request.urlopen(req) as r: return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e: return e.code, json.loads(e.read())

for f in ["high_risk.json", "low_risk.json", "partial_input.json", "invalid.json"]:
    print(f, *post("/predict", json.loads((ROOT / "samples" / f).read_text())))

In [ ]:
sh(f"{sys.executable} scripts/load_test.py --url {BASE} --requests 200")
stats = json.loads(urllib.request.urlopen(BASE + "/monitoring/stats").read())
print({k: v for k, v in stats.items() if k != "recent"})
metrics = urllib.request.urlopen(BASE + "/metrics").read().decode()
print("\n".join(l for l in metrics.splitlines() if l.startswith("cardiorisk_predictions_total")))
print("\nlast request logs:"); print("".join(open(ROOT / "api.log").readlines()[-3:]))

## 6. Optional: Kubernetes and Kubeflow Pipelines from Prayogshala

In [ ]:
if shutil.which("kubectl"):
    for verb in ["create deployments", "create services", "create pods"]:
        sh(f"kubectl auth can-i {verb}", check=False)
    print("If allowed: push the image to a registry the cluster can pull from, set it in k8s/base/deployment.yaml, then\n"
          "  kubectl apply -k k8s/base -n <your-namespace>   (namespace.yaml may need removing in a shared cluster)")
else:
    print("kubectl not available in this notebook image - deployment is done with kind/minikube or the CI deploy-kind job.")

In [ ]:
if RUN_KFP_PIPELINE:
    sh(f"{sys.executable} -m pip install -q 'kfp>=2.7,<3'")
    sh(f"{sys.executable} kubeflow/pipeline.py --compile-only")
    import kfp
    run = kfp.Client().create_run_from_pipeline_package("kubeflow/heart_disease_pipeline.yaml",
            arguments={"repo_zip_url": REPO_URL.replace(".git", "") + "/archive/refs/heads/main.zip"})
    print("submitted KFP run:", run.run_id)
else:
    print("Kubeflow pipeline skipped (set RUN_KFP_PIPELINE = True)")

## 7. Clean up background processes

In [ ]:
for p in (api, mlflow_ui):
    p.terminate()
print("stopped API and MLflow UI. Artefacts:", [str(p.relative_to(ROOT)) for p in [ROOT/"models/model.joblib", ROOT/"models/model_metadata.json", ROOT/"reports/metrics.json", ROOT/"mlflow.db"]])